# Lezione 9 · Gli LLM: token, pretraining, scaling

Il taglio delle parole con il BPE, e la curva della vasca.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

## Kata per casa · Esercizio 1: il BPE a mano, in codice

Testo: «rete rete rete rete reti reti reti remi remi». Le fusioni non scavalcano le parole.

In [2]:
from collections import Counter

parole = "rete rete rete rete reti reti reti remi remi".split()
pezzi = [list(p) for p in parole]
print("token all'inizio:", sum(len(p) for p in pezzi))

def coppie(pezzi):
    c = Counter()
    for p in pezzi:
        for a, b in zip(p, p[1:]):
            c[(a, b)] += 1
    return c

def fondi(pezzi, coppia):
    nuovi = []
    for p in pezzi:
        out, i = [], 0
        while i < len(p):
            if i < len(p) - 1 and (p[i], p[i+1]) == coppia:
                out.append(p[i] + p[i+1]); i += 2
            else:
                out.append(p[i]); i += 1
        nuovi.append(out)
    return nuovi

attesi = [(("r", "e"), 9, 27), (("re", "t"), 7, 20), (("ret", "e"), 4, 16)]
for passo, (coppia_attesa, volte_attese, token_attesi) in enumerate(attesi, 1):
    coppia, volte = coppie(pezzi).most_common(1)[0]
    pezzi = fondi(pezzi, coppia)
    totale = sum(len(p) for p in pezzi)
    print(f"fusione {passo}: {coppia[0]} + {coppia[1]} ({volte} volte) → {totale} token")
    assert coppia == coppia_attesa and volte == volte_attese and totale == token_attesi

print("prossima fusione:", coppie(pezzi).most_common(1)[0])
print(sorted(set(" · ".join(p) for p in pezzi)))

token all'inizio: 36
fusione 1: r + e (9 volte) → 27 token
fusione 2: re + t (7 volte) → 20 token
fusione 3: ret + e (4 volte) → 16 token
prossima fusione: (('ret', 'i'), 3)
['re · m · i', 'ret · i', 'rete']


## La curva della vasca

La formula dello studio del 2022, con i coefficienti riportati nel libro.

In [3]:
E, A, B, alfa, beta = 1.69, 406.4, 410.7, 0.34, 0.28

def loss(N, D):
    return E + A / N**alfa + B / D**beta

for N, atteso in [(1e8, 3.49), (1e9, 2.58), (1e10, 2.13), (1e11, 1.91), (1e12, 1.80)]:
    D = 20 * N
    print(f"{N:.0e} parametri, {D:.0e} token, calcolo {6*N*D:.1e}")
    controlla("loss prevista", loss(N, D), atteso, 0.01)

1e+08 parametri, 2e+09 token, calcolo 1.2e+18
✓  loss prevista: calcolato 3.486, nel libro 3.49
1e+09 parametri, 2e+10 token, calcolo 1.2e+20
✓  loss prevista: calcolato 2.58, nel libro 2.58
1e+10 parametri, 2e+11 token, calcolo 1.2e+22
✓  loss prevista: calcolato 2.133, nel libro 2.13
1e+11 parametri, 2e+12 token, calcolo 1.2e+24
✓  loss prevista: calcolato 1.912, nel libro 1.91
1e+12 parametri, 2e+13 token, calcolo 1.2e+26
✓  loss prevista: calcolato 1.801, nel libro 1.8


## Il duello: Gopher contro Chinchilla

Stesso calcolo, circa: la nave grande poco addestrata e quella piccola che ha letto di più.

In [4]:
gopher, chinchilla = (280e9, 300e9), (70e9, 1.4e12)
print(f"calcolo Gopher {6*gopher[0]*gopher[1]:.1e}, Chinchilla {6*chinchilla[0]*chinchilla[1]:.1e}")
print(f"loss prevista: Gopher {loss(*gopher):.3f}, Chinchilla {loss(*chinchilla):.3f}")
assert loss(*chinchilla) < loss(*gopher)
print(f"token per parametro del modello da 8 miliardi: {15e12 / 8e9:.0f}")

calcolo Gopher 5.0e+23, Chinchilla 5.9e+23
loss prevista: Gopher 1.993, Chinchilla 1.937
token per parametro del modello da 8 miliardi: 1875
